# Sistema de Recomendação de Veículos — Sauter Digital (MVP)

## Contexto do Negócio

A **Sauter Digital**, uma plataforma emergente de classificados de veículos, enfrenta um problema crítico de **retenção de usuários**. Atualmente, quando um cliente visita a página de um veículo (ex: Toyota Corolla 2020), a seção "Carros Similares" exibe apenas outros veículos idênticos — mesmo modelo, mesma marca. O resultado é previsível: se o cliente não encontra o preço ou a quilometragem ideal **naquele modelo específico**, ele abandona o site.

## A Solução Proposta

Construir o **MVP de um motor de recomendação** para uma nova seção estratégica: **"Outros que você pode gostar"** (Others you may like). O objetivo é reter o cliente apresentando **alternativas viáveis de mercado** — veículos com características técnicas e faixas de preço similares, mas obrigatoriamente de **outras marcas**.

## O Desafio Técnico: Cold Start

Como a plataforma é nova, **não existe histórico** de navegação, cliques ou compras dos usuários. Isso é um problema clássico chamado **Cold Start**. A solução adotada é a **Recomendação Baseada em Conteúdo (Content-Based Filtering)**: a inteligência do sistema é construída puramente a partir das **características físicas e de mercado dos próprios veículos**, sem depender de dados comportamentais.

## Dados Utilizados

O dataset foi extraído do Kaggle (`vagnerbessa/average-car-prices-bazil`) e contém dados da **Tabela FIPE** com as seguintes colunas: `brand`, `model`, `fuel`, `gear`, `engine_size`, `year_model`, `avg_price_brl` e `age_years`.

---

## 1. Ingestão e Limpeza de Dados

### O que acontece nesta seção?
Carregamos o dataset da FIPE e aplicamos filtros e limpezas para preparar uma base confiável.

### Decisões tomadas e justificativas:

- **Filtro Temporal (Janeiro 2022):** O dataset original contém dados mensais de todo o ano de 2022. Optamos por fixar um único mês (Janeiro) para evitar que o mesmo carro apareça 12 vezes com preços ligeiramente diferentes. Isso garante **uma entrada por veículo** e estabilidade de preços.

- **Seleção de Features:** Descartamos colunas que não agregam valor à similaridade entre veículos:
  - `year_of_reference`, `month_of_reference` → metadados temporais da tabela FIPE, não do carro
  - `fipe_code`, `authentication` → identificadores internos sem valor preditivo

- **Deduplicação:** Removemos linhas duplicadas na combinação `brand + model + gear + fuel + engine_size + age_years` para evitar que o mesmo veículo apareça mais de uma vez nas recomendações.

- **Chave de Busca (`termo_busca`):** Criamos uma coluna concatenando marca, modelo, ano e motor em texto minúsculo. Isso permite ao usuário pesquisar de forma flexível (ex: "corolla 2020" ou "toyota 1.8").

In [1]:
import pandas as pd
from IPython.display import display, HTML, clear_output

# --- Carregando o dataset completo da FIPE ---
df = pd.read_csv('fipe_2022.csv')

# --- Filtro Temporal: apenas Janeiro para evitar duplicidade mensal ---
df_base = df[df['month_of_reference'] == 'January'].copy()

# --- Seleção de features relevantes para o algoritmo ---
colunas_uteis = ['brand', 'model', 'fuel', 'gear', 'engine_size', 'year_model', 'avg_price_brl', 'age_years']
df_carros = df_base[colunas_uteis].copy().reset_index(drop=True)

# --- Deduplicação: remove veículos com características idênticas ---
df_carros = df_carros.drop_duplicates(
    subset=['brand', 'model', 'gear', 'fuel', 'engine_size', 'age_years']
).reset_index(drop=True)

# --- Chave de busca textual para o usuário pesquisar livremente ---
df_carros['termo_busca'] = (
    df_carros['brand'].astype(str) + " " + 
    df_carros['model'].astype(str) + " " + 
    df_carros['year_model'].astype(str) + " " + 
    df_carros['engine_size'].astype(str)
).str.lower()

print(f"Dataset original: {len(df)} registros")
print(f"Após filtro de Janeiro: {len(df_base)} registros")
print(f"Após deduplicação: {len(df_carros)} veículos únicos prontos para análise")

Dataset original: 290275 registros
Após filtro de Janeiro: 24031 registros
Após deduplicação: 24031 veículos únicos prontos para análise


---

## 2. Feature Engineering: Traduzindo Características em Matemática

### O que acontece nesta seção?
Transformamos as características dos veículos em uma **matriz numérica** que o algoritmo de similaridade consiga interpretar. Sem essa etapa, o computador não saberia comparar "Gasolina" com "Flex", nem entenderia que um motor 1.0 é menor que um 2.0.

### Decisões tomadas e justificativas:

- **Variáveis Numéricas** (`engine_size`, `age_years`, `avg_price_brl`):
  - Escolhemos o **RobustScaler** em vez do StandardScaler. **Por quê?** Os preços dos veículos possuem outliers extremos (min R$1.831, max R$7.695.250, mediana R$45.021). O StandardScaler usa média e desvio padrão, que são altamente distorcidos por esses extremos. O RobustScaler usa **mediana e intervalo interquartil (IQR)**, sendo resistente a outliers e preservando melhor as relações entre veículos populares.
  - **Removemos `year_model`** da lista de features numéricas. Motivo: `age_years = 2022 - year_model`, ou seja, são **linearmente dependentes**. Incluir ambas duplicaria o peso da "idade" no cálculo de similaridade, enviesando o resultado.

- **Variáveis Categóricas** (`fuel`, `gear`):
  - Aplicamos **One-Hot Encoding** para transformar categorias textuais em vetores binários (0 ou 1). Exemplo: `fuel = "Gasoline"` vira `[1, 0, 0]` se houver 3 tipos de combustível.
  - Usamos `drop='if_binary'` para evitar **multicolinearidade**: se uma variável tem apenas 2 categorias (ex: manual/automático), basta uma coluna — a outra é o complemento lógico.

### Por que essa combinação?
O pipeline `ColumnTransformer` do scikit-learn nos permite aplicar transformações diferentes para cada tipo de variável em uma única operação, mantendo o código limpo e garantindo que o escalonamento e o encoding sejam aplicados corretamente.

In [2]:
%pip install scikit-learn --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# --- Definição das features por tipo ---
vars_numericas = ['engine_size', 'age_years', 'avg_price_brl']
vars_categoricas = ['fuel', 'gear']

# --- Pipeline de transformação ---
preparador = ColumnTransformer([
    ('numeros', RobustScaler(), vars_numericas),
    ('categorias', OneHotEncoder(drop='if_binary', handle_unknown='ignore'), vars_categoricas)
])

# --- Gerando a matriz matemática ("DNA" de cada veículo) ---
matriz_carros = preparador.fit_transform(df_carros)

print(f"Matriz matemática gerada com sucesso!")
print(f"Dimensões: {matriz_carros.shape[0]} veículos x {matriz_carros.shape[1]} features numéricas")
print(f"\nFeatures numéricas escalonadas: {vars_numericas}")
print(f"Features categóricas codificadas: {vars_categoricas}")

Matriz matemática gerada com sucesso!
Dimensões: 24031 veículos x 7 features numéricas

Features numéricas escalonadas: ['engine_size', 'age_years', 'avg_price_brl']
Features categóricas codificadas: ['fuel', 'gear']


---

## 3. O Motor de Recomendação: Similaridade de Cosseno + Regras de Negócio

### O que acontece nesta seção?
Aqui definimos a **função principal** do sistema: dado um veículo escolhido pelo usuário, o algoritmo calcula quais outros veículos são mais "parecidos" e aplica filtros de negócio antes de retornar as recomendações.

### O Motor Matemático: Similaridade de Cosseno

Escolhemos a **Similaridade de Cosseno** como métrica de comparação. Ela calcula o **ângulo entre dois vetores** no espaço multidimensional das features. Quanto mais próximo de **1.0**, mais semelhantes são os veículos.

**Por que Cosseno e não Distância Euclidiana?**
- A distância euclidiana mede a "distância absoluta" entre dois pontos — ela é afetada pela **magnitude** dos vetores. Se um carro tem um preço muito alto, ele ficaria "longe" de todos os outros, mesmo sendo tecnicamente parecido.
- A similaridade de cosseno mede a **direção** dos vetores, ignorando a magnitude. Isso significa que ela captura o **perfil** do veículo (proporção entre motor, preço, idade) independentemente da escala absoluta.
- Combinada com o RobustScaler, temos uma dupla camada de proteção contra distorções de escala.

### Regras de Negócio (Filtros de Saída)

O algoritmo não entrega apenas o que é matematicamente parecido — ele aplica **filtros estratégicos**:

1. **Priorização de Marcas Concorrentes:** O algoritmo **prioriza** recomendações de outras marcas, preenchendo o carrossel primeiro com concorrentes. Se o cliente busca um Toyota, ele verá prioritariamente Honda, Chevrolet, Hyundai. Porém, caso não existam concorrentes suficientes na faixa de preço, o sistema **completa** as vagas restantes com veículos da mesma marca — garantindo que o carrossel nunca fique vazio ou incompleto. Essa abordagem é mais realista do que um bloqueio total: em nichos com poucas marcas (ex: caminhões pesados), um bloqueio absoluto poderia resultar em zero recomendações.

2. **Coerência de Preço (±25%):** Uma trava lógica que garante que as recomendações estejam dentro de uma **margem financeira realista**. Um carro de R$50.000 nunca será sugerido para quem busca um de R$300.000 (e vice-versa).

3. **Ranking por Similaridade:** Os resultados são ordenados do mais similar ao menos similar, exibindo o **Top N** (padrão: 10).

In [4]:
from sklearn.metrics.pairwise import cosine_similarity

def buscar_concorrentes_semelhantes(id_carro, top_n=10, margem_preco=0.25):
    """
    Dado o índice de um veículo, retorna os top_n veículos mais similares
    dentro da margem de preço, PRIORIZANDO outras marcas.
    Se não houver concorrentes suficientes, completa com a mesma marca.
    """
    # Calcula a similaridade do veículo escolhido contra TODOS os outros
    vetor_escolhido = matriz_carros[id_carro].reshape(1, -1)
    similaridades = cosine_similarity(vetor_escolhido, matriz_carros).flatten()
    
    # Monta o dataframe de resultados com os scores
    df_resultado = df_carros.copy()
    df_resultado['score_similaridade'] = similaridades
    
    # Recupera os dados do carro original para aplicar os filtros
    carro_original = df_carros.iloc[id_carro]
    marca_original = carro_original['brand']
    preco_original = carro_original['avg_price_brl']
    
    # --- REGRA 1: Margem de preço de ±25% (coerência financeira) ---
    preco_minimo = preco_original * (1 - margem_preco)
    preco_maximo = preco_original * (1 + margem_preco)
    filtro_preco_justo = df_resultado['avg_price_brl'].between(preco_minimo, preco_maximo)
    
    # Exclui o próprio carro do resultado
    candidatos = df_resultado[filtro_preco_justo & (df_resultado.index != id_carro)].sort_values(
        by='score_similaridade', ascending=False
    )
    
    # --- REGRA 2: Priorizar outras marcas, completar com mesma marca ---
    outras_marcas = candidatos[candidatos['brand'] != marca_original].head(top_n)
    
    if len(outras_marcas) < top_n:
        # Completa as vagas restantes com veículos da mesma marca
        vagas_restantes = top_n - len(outras_marcas)
        mesma_marca = candidatos[candidatos['brand'] == marca_original].head(vagas_restantes)
        recomendacoes = pd.concat([outras_marcas, mesma_marca]).sort_values(
            by='score_similaridade', ascending=False
        )
    else:
        recomendacoes = outras_marcas
    
    return carro_original, recomendacoes.head(top_n)

print("Motor de recomendação carregado com sucesso!")

Motor de recomendação carregado com sucesso!


---

## 4. Interface Interativa: Consultor Automotivo

### O que acontece nesta seção?
Essa é a **camada de apresentação** do MVP — a interface com a qual o usuário interage. Ao executar esta célula, o sistema:

1. Solicita ao usuário que digite uma busca livre (marca, modelo, ano ou motor)
2. Localiza os veículos correspondentes usando a chave `termo_busca`
3. Apresenta uma lista numerada dos resultados encontrados
4. O usuário seleciona o veículo exato pelo número
5. O motor de recomendação é acionado e exibe um **painel visual** com:
   - Os dados do carro de interesse
   - Uma tabela estilizada com as **Top 10 alternativas similares**, priorizando marcas concorrentes

### Decisões de UX:
- **Busca flexível por termos:** O usuário pode digitar "corolla", "toyota 2020", "1.8" — o sistema filtra progressivamente por cada termo digitado.
- **Limite de 15 resultados na busca:** Evita sobrecarregar o usuário com centenas de opções.
- **Painel HTML estilizado:** Simula a experiência visual que o carrossel "Others you may like" teria na plataforma real.

In [5]:
print("=" * 70)
print(" BEM-VINDO AO CONSULTOR AUTOMOTIVO SAUTER DIGITAL ")
print("=" * 70)

pesquisa = input("Digite a marca, modelo, ano ou motor do carro que você gosta (ex: 'toyota corolla 2020'): ").lower()

termos = pesquisa.split()
resultados_busca = df_carros.copy()

for termo in termos:
    resultados_busca = resultados_busca[resultados_busca['termo_busca'].str.contains(termo, na=False)]

if resultados_busca.empty:
    print(f"\nNenhum carro encontrado com os termos: '{pesquisa}'. Tente ser mais genérico.")
else:
    clear_output(wait=True)
    display(HTML("<h3>Encontramos estes modelos. Digite o NÚMERO do carro exato que você quer analisar:</h3>"))
    
    opcoes_exibicao = resultados_busca.head(15)
    for i, (idx, row) in enumerate(opcoes_exibicao.iterrows()):
        print(f" [{i}] {row['brand']} {row['model']} ({row['year_model']}) - Motor: {row['engine_size']} - Preço: R$ {row['avg_price_brl']:,.2f}")
    
    try:
        escolha_idx = int(input("\nSeu número escolhido: "))
        id_real_do_carro = opcoes_exibicao.index[escolha_idx]
        
        carro_alvo, concorrentes = buscar_concorrentes_semelhantes(id_real_do_carro, top_n=10)
        
        clear_output(wait=True)
        
        painel_html = f"""
        <div style="background-color: #2c3e50; padding: 20px; border-radius: 10px; color: white; font-family: Arial;">
            <h2 style="margin-top:0; color: #f1c40f;">SEU CARRO DE INTERESSE</h2>
            <p style="font-size: 18px; margin: 5px 0;"><b>{carro_alvo['brand']} - {carro_alvo['model']} ({carro_alvo['year_model']})</b></p>
            <hr style="border-color: #34495e;">
            <div style="display: flex; gap: 40px; font-size: 15px; margin-top: 10px;">
                <span><b>Preço:</b> R$ {carro_alvo['avg_price_brl']:,.2f}</span>
                <span><b>Motor:</b> {carro_alvo['engine_size']}</span>
                <span><b>Combustível:</b> {carro_alvo['fuel']}</span>
            </div>
        </div>
        <h3 style="color: #2c3e50; margin-top: 30px;">Outros que você pode gostar (priorizando marcas concorrentes):</h3>
        """
        display(HTML(painel_html))
        
        tabela_final = concorrentes[['brand', 'model', 'year_model', 'engine_size', 'fuel', 'avg_price_brl', 'score_similaridade']].copy()
        tabela_final.columns = ['Marca', 'Modelo', 'Ano', 'Motor', 'Combustível', 'Preço (R$)', 'Similaridade']
        
        tabela_estilizada = tabela_final.style.format({
            'Preço (R$)': 'R$ {:,.2f}',
            'Similaridade': '{:.1%}'
        }).bar(subset=['Similaridade'], color='#27ae60', vmin=0, vmax=1)\
          .hide(axis='index')\
          .set_properties(**{'text-align': 'left', 'padding': '10px', 'border-bottom': '1px solid #ddd'})
          
        display(tabela_estilizada)

    except (ValueError, IndexError):
        print("\nNúmero inválido. Por favor, rode a célula novamente e digite um número válido da lista.")

Marca,Modelo,Ano,Motor,Combustível,Preço (R$),Similaridade
Mitsubishi,OUTLANDER Sport HPE 4x4 2.0 16V Flex Aut,2020,2.000000,Gasoline,"R$ 130,169.00",100.0%
Honda,Civic Sedan EXL 2.0 Flex 16V Aut.4p,2020,2.000000,Gasoline,"R$ 128,471.00",100.0%
Jeep,COMPASS LONGITUDE 2.0 4x2 Flex 16V Aut.,2020,2.000000,Gasoline,"R$ 135,768.00",100.0%
Jeep,COMPASS SPORT 2.0 4x2 Flex 16V Aut.,2020,2.000000,Gasoline,"R$ 126,012.00",100.0%
Hyundai,ix35 GLS 2.0 16V 2WD Flex Aut.,2021,2.000000,Gasoline,"R$ 135,659.00",99.9%
Kia Motors,Sportage LX 2.0 16V/ 2.0 16V Flex Aut.,2020,2.000000,Gasoline,"R$ 137,301.00",99.9%
Jeep,COMPASS LONGITUDE 2.0 4x2 Flex 16V Aut.,2019,2.000000,Gasoline,"R$ 129,018.00",99.9%
Volvo,V40 T-4 KINETIC 2.0 Aut.,2019,2.000000,Gasoline,"R$ 129,721.00",99.9%
Jeep,COMPASS SPORT 2.0 4x2 Flex 16V Aut.,2021,2.000000,Gasoline,"R$ 136,665.00",99.9%
Honda,Civic Sedan EX 2.0 Flex 16V Aut.4p,2021,2.000000,Gasoline,"R$ 130,447.00",99.9%
